# V.E.R.D.E. · Treinar o detector de resíduos no Google Colab (grátis)

Este notebook treina o **detector** com o dataset **TACO** (1.500 fotos de lixo real, 60 classes convertidas
para as 5 categorias da coleta seletiva) numa GPU gratuita do Google. Nada roda no seu computador e nada é pago.

**Antes de começar:** menu **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4** → Salvar.

Tempo aproximado na T4: 1 a 2 horas para 100 épocas. Ao final, você baixa o arquivo `waste_detector.pt` e coloca na
pasta `models/` do projeto: o V.E.R.D.E. passa a usá-lo sozinho.

## 1. Conferir a GPU
Deve aparecer uma **Tesla T4**. Se aparecer erro, volte ao passo "Antes de começar".

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Salvar o progresso no Google Drive (recomendado)
O Colab gratuito pode desconectar. Com o Drive montado, o treino e o modelo final ficam salvos em
`Meu Drive/VERDE`. Autorize o acesso quando pedir.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUNS = '/content/drive/MyDrive/VERDE/runs'
!mkdir -p "$RUNS"

## 3. Baixar o projeto
O repositório é privado, então o Colab precisa de um **token do GitHub** só de leitura:
github.com → Settings → Developer settings → Personal access tokens → **Fine-grained tokens** → Generate new token →
Repository access: *Only select repositories* → `V.E.R.D.E` → Permissions: **Contents: Read-only** → Generate.
Cole o token quando pedir (ele não fica salvo no notebook).

In [ ]:
from getpass import getpass
token = getpass('Token do GitHub (só leitura): ')
!git clone -q https://{token}@github.com/ThurCintra/V.E.R.D.E.git /content/VERDE
del token
%cd /content/VERDE
!git log --oneline -1

## 4. Instalar as dependências (2 a 3 minutos)

In [ ]:
!pip install -q -r requirements.txt
!python scripts/download_model.py yolov8s.pt

## 5. Treinar o detector com o TACO
Baixa o TACO (anotações do GitHub, fotos do Flickr), converte para as 5 categorias e treina a partir do `yolov8s`
(mais preciso que o `yolov8n`). Algumas fotos antigas do Flickr podem ter saído do ar: o script avisa e segue
sem elas.

In [ ]:
!python -m src.train taco --base models/yolov8s.pt --epochs 100 --imgsz 640 --batch 16 --device 0 --runs-dir "$RUNS"
!cp models/waste_detector.pt "$RUNS/../waste_detector.pt"

## 6. Avaliar o resultado
`mAP50` é a nota principal do detector (0 a 1; acima de 0,5 já é bom para lixo real, que é um problema difícil).

In [ ]:
!python -m src.evaluate detector --models models/waste_detector.pt --data datasets/taco/data.yaml
!cat reports/detectores.json

## 7. (Opcional) Classificador de material maior
O projeto já vem com um classificador treinado em CPU. Na GPU dá para treinar a versão `s` por mais épocas:

In [ ]:
!python scripts/download_model.py yolov8s-cls.pt
!python -m src.train trashnet --base models/yolov8s-cls.pt --epochs 60 --device 0 --runs-dir "$RUNS"
!python -m src.evaluate material
!cp models/material_cls.pt "$RUNS/../material_cls.pt"

## 8. Baixar os modelos
Os arquivos também ficam em `Meu Drive/VERDE`. No seu PC, coloque-os na pasta `models/` do projeto e reinicie o
dashboard e a API.

In [ ]:
from google.colab import files
files.download('models/waste_detector.pt')